In [6]:
import pandas as pd

In [7]:
nsw_cleaned = pd.read_csv("nsw_cleaned.csv")
public_holidays = pd.read_csv("public_holidays.csv")
school_terms = pd.read_csv("school_terms.csv")
solar_radiation_bankstown = pd.read_csv("solar_radiation_bankstown_66137.csv")

print(nsw_cleaned.shape)
print(public_holidays.shape)
print(school_terms.shape)
print(solar_radiation_bankstown.shape)

(196513, 8)
(141, 2)
(48, 5)
(4095, 14)


In [8]:
nsw_cleaned["DATETIME"] = pd.to_datetime(nsw_cleaned["DATETIME"])
nsw_cleaned["year"] = nsw_cleaned["DATETIME"].dt.year
nsw_cleaned["month"] = nsw_cleaned["DATETIME"].dt.month
nsw_cleaned["day"] = nsw_cleaned["DATETIME"].dt.day

nsw_cleaned.head()

,DATETIME,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,forecast_closest,forecast_12hr_prior,forecast_dayprior,year,month,day
0,2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.1,7999.11,7811.86,7822.38,2010,1,1
1,2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.9,7596.21,7612.63,7715.68,2010,1,1
2,2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.6,7380.70,7339.64,7482.56,2010,1,1
3,2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.5,7022.05,7009.55,7129.32,2010,1,1
4,2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.5,6682.92,6683.15,6800.73,2010,1,1


In [9]:
import numpy as np

# keeping just teh date
nsw_cleaned["date"] = nsw_cleaned["DATETIME"].dt.normalize()

# kellys solar dariation, just joining it on date, so a given day will get the same raditaion all day
solar = solar_radiation_bankstown[["YYYY-MM-DD", "radiation"]].copy()
solar["date"] = pd.to_datetime(solar["YYYY-MM-DD"], dayfirst=True)
solar = solar[["date", "radiation"]].dropna(subset=["date"])

nsw_cleaned = nsw_cleaned.merge(solar, on="date", how="left")

# yes no public holiday flag
holidays = public_holidays.copy()
holidays["date"] = pd.to_datetime(holidays["date"])
holiday_dates = set(holidays["date"])

nsw_cleaned["is_public_holiday"] = nsw_cleaned["date"].isin(holiday_dates)

# yes no school holiday flag
terms = school_terms.copy()
terms["start_date"] = pd.to_datetime(terms["start_date"])
terms["end_date"] = pd.to_datetime(terms["end_date"])

def in_school_term(d):
    return bool(((terms["start_date"] <= d) & (terms["end_date"] >= d)).any())

unique_dates = nsw_cleaned["date"].unique()
term_flags = {d: in_school_term(pd.Timestamp(d)) for d in unique_dates}
nsw_cleaned["is_school_term"] = nsw_cleaned["date"].map(term_flags)

nsw_cleaned.drop(columns=["date"], inplace=True)

nsw_cleaned.head()

,DATETIME,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,forecast_closest,forecast_12hr_prior,forecast_dayprior,year,month,day,radiation,is_public_holiday,is_school_term
0,2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.1,7999.11,7811.86,7822.38,2010,1,1,14.6,True,False
1,2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.9,7596.21,7612.63,7715.68,2010,1,1,14.6,True,False
2,2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.6,7380.70,7339.64,7482.56,2010,1,1,14.6,True,False
3,2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.5,7022.05,7009.55,7129.32,2010,1,1,14.6,True,False
4,2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.5,6682.92,6683.15,6800.73,2010,1,1,14.6,True,False


In [10]:
nsw_cleaned["day_of_week"] = nsw_cleaned["DATETIME"].dt.day_name()

nsw_cleaned.head()

,DATETIME,TOTALDEMAND,REGIONID,LOCATION,TEMPERATURE,forecast_closest,forecast_12hr_prior,forecast_dayprior,year,month,day,radiation,is_public_holiday,is_school_term,day_of_week
0,2010-01-01 00:00:00,8038.00,NSW1,Bankstown,23.1,7999.11,7811.86,7822.38,2010,1,1,14.6,True,False,Friday
1,2010-01-01 00:30:00,7809.31,NSW1,Bankstown,22.9,7596.21,7612.63,7715.68,2010,1,1,14.6,True,False,Friday
2,2010-01-01 01:00:00,7483.69,NSW1,Bankstown,22.6,7380.70,7339.64,7482.56,2010,1,1,14.6,True,False,Friday
3,2010-01-01 01:30:00,7117.23,NSW1,Bankstown,22.5,7022.05,7009.55,7129.32,2010,1,1,14.6,True,False,Friday
4,2010-01-01 02:00:00,6812.03,NSW1,Bankstown,22.5,6682.92,6683.15,6800.73,2010,1,1,14.6,True,False,Friday


In [11]:
nsw_cleaned.to_csv("nsw_features_added.csv", index=False)

In [13]:
nsw_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 196513 entries, 0 to 196512
Data columns (total 15 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   DATETIME             196513 non-null  datetime64[ns]
 1   TOTALDEMAND          196513 non-null  float64       
 2   REGIONID             196513 non-null  object        
 3   LOCATION             196513 non-null  object        
 4   TEMPERATURE          196513 non-null  float64       
 5   forecast_closest     196513 non-null  float64       
 6   forecast_12hr_prior  196513 non-null  float64       
 7   forecast_dayprior    196513 non-null  float64       
 8   year                 196513 non-null  int32         
 9   month                196513 non-null  int32         
 10  day                  196513 non-null  int32         
 11  radiation            196513 non-null  float64       
 12  is_public_holiday    196513 non-null  bool          
 13  is_school_term